The goal of this section is to identify frequent product associations in customer orders using Association Rule Mining. The objective is to generate rules of the form {A} → {B}, where the presence of item A is associated with the presence of item B in the same order. Such rules can be useful for understanding purchasing patterns and can support applications such as product recommendations and cross-selling.

The analysis begins by identifying products that occur frequently within customer orders and then generating pairs of products that appear together. A minimum support threshold is used to remove infrequent products and item pairs, reducing the number of combinations that need to be analysed and making the process more computationally efficient.

Once the frequent item pairs have been identified, three key metrics are used to evaluate the resulting associations: support, confidence, and lift.

1. Support

Support measures how frequently an item or item pair occurs across all orders. For two items A and B, it represents the percentage of orders that contain both items:

support{A,B} = number of orders containing A and B / total number of orders

A higher support indicates that the combination occurs more frequently in the dataset.

2. Confidence

Confidence measures how often item B occurs in an order given that item A is present. It is directional, meaning that confidence for A → B can be different from confidence for B → A:

confidence{A→B} = support{A,B} / support{A}

For example, a high confidence value for A → B indicates that customers who purchase A frequently also purchase B.

3. Lift

Lift measures the strength of the relationship between two items by comparing their observed co-occurrence with the co-occurrence that would be expected based on their individual frequencies:

lift{A,B} = support{A,B} / (support{A} × support{B})

Unlike confidence, lift is not directional, so lift{A,B} = lift{B,A}. Its value can be interpreted as follows:

Lift = 1: A and B occur together approximately as often as expected by chance.
Lift > 1: A and B have a positive association and occur together more frequently than expected.
Lift < 1: A and B have a negative association and occur together less frequently than expected.

In [ ]:
import pandas as pd
import numpy as np
import sys
from itertools import combinations, groupby
from collections import Counter
from IPython.display import display

In [ ]:
# Function that returns the size of an object in MB
def size(obj):
    return "{0:.2f} MB".format(sys.getsizeof(obj) / (1000 * 1000))

Data Preparation

In [ ]:
orders = pd.read_csv('order_products__prior.csv')
print('orders -- dimensions: {0};   size: {1}'.format(orders.shape, size(orders)))
display(orders.head())

orders -- dimensions: (28513580, 4);   size: 912.43 MB


,order_id,product_id,add_to_cart_order,reordered
0,2,33120.0,1.0,1.0
1,2,28985.0,2.0,1.0
2,2,9327.0,3.0,0.0
3,2,45918.0,4.0,1.0
4,2,30035.0,5.0,0.0




Convert order data into format expected by the association rules function

In [ ]:
# Convert from DataFrame to a Series, with order_id as index and item_id as value
orders = orders.set_index('order_id')['product_id'].rename('item_id')
display(orders.head(10))
type(orders)

,item_id
order_id,
2,33120.0
2,28985.0
2,9327.0
2,45918.0
2,30035.0
2,17794.0
2,40141.0
2,1819.0
2,43668.0


pandas.core.series.Series

Display summary statistics for order data

In [ ]:
print('dimensions: {0};   size: {1};   unique_orders: {2};   unique_items: {3}'
      .format(orders.shape, size(orders), len(orders.index.unique()), len(orders.value_counts())))

dimensions: (28513580,);   size: 456.22 MB;   unique_orders: 2825829;   unique_items: 49650


Association Rules Function

In [ ]:
from collections import Counter, defaultdict
from itertools import combinations, groupby
import pandas as pd

# Defining helper functions
def freq(generator):
    counts = defaultdict(int)
    for item in generator:
        counts[item] += 1
    return pd.Series(counts).rename("freq")

def order_count(order_item):
    return len(set(order_item.index))

def get_item_pairs(order_item):
    order_item = order_item.reset_index().values
    for order_id, order_object in groupby(order_item, lambda x: x[0]):
        item_list = [item[1] for item in order_object]
        for item_pair in combinations(item_list, 2):
            yield tuple(sorted(item_pair))  # sort to ensure (A,B) = (B,A)

def merge_item_stats(item_pairs, item_stats):
    return (item_pairs
                .merge(item_stats.rename(columns={'freq': 'freqA', 'support': 'supportA'}), left_on='item_A', right_index=True)
                .merge(item_stats.rename(columns={'freq': 'freqB', 'support': 'supportB'}), left_on='item_B', right_index=True))

def merge_item_name(rules, item_name):
    columns = ['itemA','itemB','freqAB','supportAB','freqA','supportA','freqB','supportB',
               'confidenceAtoB','confidenceBtoA','lift']
    rules = (rules
                .merge(item_name.rename(columns={'item_name': 'itemA'}), left_on='item_A', right_on='item_id')
                .merge(item_name.rename(columns={'item_name': 'itemB'}), left_on='item_B', right_on='item_id'))
    return rules[columns]

# Load and convert orders to expected format (order_id as index, product_id as values)
orders_df = pd.read_csv('orders.csv')
prior = pd.read_csv('order_products__prior.csv')
orders = prior.merge(orders_df[['order_id']], on='order_id')
orders = orders.set_index('order_id')['product_id'].rename('item_id')  # Series format

# Association rule mining function
def association_rules(order_item, min_support):

    print("Starting order_item: {:22d}".format(len(order_item)))

    item_stats = freq(order_item).to_frame("freq")
    item_stats['support'] = item_stats['freq'] / order_count(order_item) * 100

    qualifying_items = item_stats[item_stats['support'] >= min_support].index
    order_item = order_item[order_item.isin(qualifying_items)]

    print("Items with support >= {}: {:15d}".format(min_support, len(qualifying_items)))
    print("Remaining order_item: {:21d}".format(len(order_item)))

    order_size = freq(order_item.index)
    qualifying_orders = order_size[order_size >= 2].index
    order_item = order_item[order_item.index.isin(qualifying_orders)]

    print("Remaining orders with 2+ items: {:11d}".format(len(qualifying_orders)))
    print("Remaining order_item: {:21d}".format(len(order_item)))

    item_stats = freq(order_item).to_frame("freq")
    item_stats['support'] = item_stats['freq'] / order_count(order_item) * 100

    # Reduce to most frequent items to save memory
    top_items = item_stats.sort_values("support", ascending=False).head(1000).index
    order_item = order_item[order_item.isin(top_items)]

    item_pair_gen = get_item_pairs(order_item)

    item_pairs = freq(item_pair_gen).to_frame("freqAB")
    item_pairs['supportAB'] = item_pairs['freqAB'] / len(qualifying_orders) * 100

    print("Item pairs: {:31d}".format(len(item_pairs)))

    item_pairs = item_pairs[item_pairs['supportAB'] >= min_support]

    print("Item pairs with support >= {}: {:10d}\n".format(min_support, len(item_pairs)))

    item_pairs.index = pd.MultiIndex.from_tuples(item_pairs.index, names=['item_A', 'item_B'])
    item_pairs = item_pairs.reset_index()

    item_pairs = merge_item_stats(item_pairs, item_stats)

    item_pairs['confidenceAtoB'] = item_pairs['supportAB'] / item_pairs['supportA']
    item_pairs['confidenceBtoA'] = item_pairs['supportAB'] / item_pairs['supportB']
    item_pairs['lift'] = item_pairs['supportAB'] / (item_pairs['supportA'] * item_pairs['supportB'])

    return item_pairs.sort_values('lift', ascending=False)

Association Rule Mining

In [ ]:
# Run association rule mining
import time

start = time.time()
rules = association_rules(orders, min_support=0.01)
end = time.time()
print(f"Execution time: {end - start:.2f} seconds")

# Load item names and merge with rules
item_name = pd.read_csv('products.csv')
item_name = item_name.rename(columns={'product_id': 'item_id', 'product_name': 'item_name'})

rules_final = merge_item_name(rules, item_name)
display(rules_final.head(10))

Starting order_item:               29612381
Items with support >= 0.01:           10901
Remaining order_item:              27246297
Remaining orders with 2+ items:     2750656
Remaining order_item:              27081149
Item pairs:                          476432
Item pairs with support >= 0.01:      49528

Execution time: 153.85 seconds


,itemA,itemB,freqAB,supportAB,freqA,supportA,freqB,supportB,confidenceAtoB,confidenceBtoA,lift
0,Yotoddler Organic Pear Spinach Mango Yogurt,Organic Whole Milk Strawberry Beet Berry Yogur...,2599,0.094487,5613,0.204060,5774,0.209914,0.463032,0.450121,2.205823
1,Vanilla Almond Milk Yogurt,Almond Milk Strawberry Yogurt,1648,0.059913,4934,0.179375,5217,0.189664,0.334009,0.315890,1.761057
2,Blueberry on the Bottom Nonfat Greek Yogurt,Strawberry on the Bottom Nonfat Greek Yogurt,2529,0.091942,5813,0.211331,6937,0.252194,0.435059,0.364567,1.725095
3,Organic Greek Nonfat Yogurt With Mixed Berries,Organic Nonfat Greek Yogurt With Peaches,2014,0.073219,6267,0.227837,5529,0.201007,0.321366,0.364261,1.598783
4,Cherry Pie Fruit & Nut Bar,Apple Pie Fruit & Nut Food Bar,1841,0.066929,5047,0.183484,6847,0.248922,0.364771,0.268877,1.465401
5,Strawberry Rhubarb Yogurt,Mixed Berries Whole Milk Icelandic Style Skyr ...,2112,0.076782,6056,0.220166,6621,0.240706,0.348745,0.318985,1.448841
6,Blueberry on the Bottom Nonfat Greek Yogurt,Peach on the Bottom Nonfat Greek Yogurt,2162,0.078599,5813,0.211331,7062,0.256739,0.371925,0.306146,1.448652
7,Peach on the Bottom Nonfat Greek Yogurt,Strawberry on the Bottom Nonfat Greek Yogurt,2420,0.087979,7062,0.256739,6937,0.252194,0.342679,0.348854,1.358790
8,Broccoli & Apple Stage 2 Baby Food,Spinach Peas & Pear Stage 2 Baby Food,2171,0.078927,6280,0.228309,7356,0.267427,0.345701,0.295133,1.292691
9,Blueberry Muffin Bar,Cherry Pie Fruit & Nut Bar,1245,0.045262,5275,0.191772,5047,0.183484,0.236019,0.246681,1.286322


Conclusion

From the output above, we see that the top associations are not surprising, with one flavor of an item being purchased with another flavor from the same item family. As mentioned, one common application of association rules mining is in the domain of recommender systems. Once item pairs have been identified as having positive relationship, recommendations can be made to customers in order to increase sales. And hopefully, along the way, also introduce customers to items they never would have tried before or even imagined existed!